# Lab 03: LoRA fine-tuning a small open model with PEFT and TRL

Applied AI Engineering, Studio 5, week 3. You will fine-tune **Qwen/Qwen2.5-0.5B-Instruct** (Apache 2.0) with LoRA
to triage support tickets into a fixed JSON format, and you will **measure** the result instead of trusting the loss.

The run has five checkpoints, and each one has a check:

1. Load data and confirm your chat formatting matches the tokenizer's template.
2. Score the **base** model on held-out tickets (your baseline).
3. Configure LoRA and confirm the trainable parameter count matches your own arithmetic.
4. Train, then read the loss curves with your `diagnose` function.
5. Score the **fine-tuned** model on the same held-out tickets, and read the failures.

**Runtime:** in Colab choose *Runtime > Change runtime type > GPU*. GPU types on the free tier vary and are not
guaranteed, so the notebook detects what it got. Training takes a few minutes on a small GPU. Nothing here needs an
API key, and nothing is pushed to the Hugging Face Hub.

Versions were pinned on the review date (2026-11-11). If an install or API call fails, check the TRL and PEFT docs for
changes before changing the logic.

In [ ]:
%pip install -q "trl==1.17.1" "peft==0.21.2" datasets matplotlib

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. In Colab: Runtime > Change runtime type > GPU, then rerun."
print(torch.cuda.get_device_name(0))
BF16_OK = torch.cuda.is_bf16_supported()
print("bf16 supported:", BF16_OK, "(older GPUs such as the T4 fall back to fp16 mixed precision)")

## 1. Get the lab code and data

The helpers you wrote in `starter.py` (formatting, parameter counts, log parsing, scoring) are imported from the repo.
This cell imports the reference `solution.py` so the notebook runs end to end. Once your offline tests pass, change
the import to `starter` and rerun: your own code then drives the run.

In [ ]:
import json, pathlib, subprocess, sys

REPO = "https://github.com/smerritt89-cl/applied-ai-engineering"
LAB = "labs/studio-5/03-lora-finetune"
if not pathlib.Path("aae").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "aae"], check=True)
    subprocess.run(["git", "-C", "aae", "sparse-checkout", "set", LAB], check=True)
sys.path.insert(0, f"aae/{LAB}")

import solution as lab   # switch to: import starter as lab

def load_jsonl(path):
    return [json.loads(line) for line in pathlib.Path(path).read_text().splitlines() if line.strip()]

raw = {split: load_jsonl(f"aae/{LAB}/data/{split}.jsonl") for split in ("train", "valid", "test")}
print({k: len(v) for k, v in raw.items()})
print(raw["train"][0])

## 2. Format the data, and check the template

TRL's `SFTTrainer` accepts a *conversational prompt-completion* dataset: a `prompt` list of messages and a
`completion` list. For that format it computes loss on the completion only by default, which is the masking you
studied in lesson 7.

Before training, confirm that the text your code thinks is trained is what the tokenizer actually renders. If your
`render_chatml` disagrees with `apply_chat_template`, your mental model of the training data is wrong.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

ds = {split: Dataset.from_list([lab.to_prompt_completion(r) for r in rows]) for split, rows in raw.items()}

for ex in ds["train"].select(range(5)):
    ours = lab.render_chatml(ex["prompt"] + ex["completion"])
    theirs = tokenizer.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
    assert ours == theirs, (ours, theirs)

prompt_text, completion_text = lab.split_prompt_completion(ds["train"][0])
print("PROMPT (masked, no loss):\n" + prompt_text)
print("COMPLETION (trained):\n" + completion_text)

## 3. Baseline: score the base model first

A fine-tuning result means nothing without a baseline. Score the untouched model on the **test** split, which uses
ticket templates that never appear in training. Decoding is greedy so reruns are comparable.

In [ ]:
from transformers import AutoModelForCausalLM

DTYPE = torch.bfloat16 if BF16_OK else torch.float16

def generate_answers(model, examples, max_new_tokens=80):
    model.eval()
    outs = []
    for ex in examples:
        text = tokenizer.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
        outs.append(tokenizer.decode(ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    return outs

def evaluate(model, split="test"):
    examples = list(ds[split])
    answers = generate_answers(model, examples)
    scores = [lab.score_prediction(a, r["label"]) for a, r in zip(answers, raw[split])]
    return lab.summarize_scores(scores), answers, scores

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="cuda")
base_summary, base_answers, _ = evaluate(base)
print("base model:", base_summary)
for a in base_answers[:3]:
    print("---\n" + a)
del base
torch.cuda.empty_cache()

## 4. Configure LoRA and predict the parameter count

`target_modules="all-linear"` puts an adapter on every linear layer in each block (attention and MLP), which the QLoRA
paper and later studies found matters for matching full fine-tuning. Before building the trainer, predict the count
with your own `lora_trainable_params`. Then compare it with what PEFT reports.

In [ ]:
from peft import LoraConfig
from transformers import AutoConfig

RANK, ALPHA = 16, 32
peft_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,           # scale = alpha / r = 2.0
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model_config = AutoConfig.from_pretrained(MODEL_ID).to_dict()
predicted = lab.lora_trainable_params(model_config, r=RANK)
print(f"predicted trainable parameters: {predicted:,}")

## 5. Train with SFTTrainer

Settings, and why:

- `learning_rate=2e-4`: adapters usually want a higher rate than full fine-tuning (TRL's docs suggest about 1e-4 for
  adapters; its full fine-tuning default is 2e-5). Try 5e-5 and 5e-4 later and compare the curves.
- `num_train_epochs=2` with evaluation every 10 steps, so you can see whether the eval loss turns upward.
- Mixed precision: bf16 where the GPU supports it, fp16 otherwise. Set both flags explicitly, because SFTConfig turns
  bf16 on by default when fp16 is not set.
- `report_to="none"`: no external logging service. `save_strategy="no"`: the adapter is saved by hand at the end.

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen25-triage-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch size 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    max_length=512,
    bf16=BF16_OK,
    fp16=not BF16_OK,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=MODEL_ID,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"PEFT trainable: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")
assert trainable == predicted, "your count and PEFT's disagree: check which modules were targeted"

In [ ]:
train_result = trainer.train()
print(train_result)

## 6. Read the loss curves

Parse `trainer.state.log_history` with your own function and plot it. Then answer, in writing, before moving on: did
the eval loss bottom out and rise? Where is the gap between train and eval? Is there anything non-finite?

In [ ]:
import matplotlib.pyplot as plt

train_pts, eval_pts = lab.parse_log_history(trainer.state.log_history)
report = lab.diagnose(train_pts, eval_pts, patience=2, min_delta=0.01)
print(report)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(*zip(*train_pts), label="train loss (completion tokens)")
if eval_pts:
    ax.plot(*zip(*eval_pts), marker="o", label="eval loss (valid split)")
ax.set_xlabel("optimizer step"); ax.set_ylabel("mean cross-entropy (nats per token)"); ax.legend()
plt.show()

## 7. Score the fine-tuned model on the same test set

Same examples, same decoding, same scorer. Then read the failures yourself: a field accuracy number tells you how
often, the failures tell you why.

In [ ]:
tuned_summary, tuned_answers, tuned_scores = evaluate(trainer.model)
print("base: ", base_summary)
print("tuned:", tuned_summary)

failures = [(r, a) for r, a, s in zip(raw["test"], tuned_answers, tuned_scores) if not s["exact"]]
print(f"{len(failures)} test tickets not exactly right")
for r, a in failures[:10]:
    print("---")
    print("ticket:", r["ticket"])
    print("gold:  ", lab.label_to_json(r["label"]))
    print("model: ", a)

## 8. Save the adapter (not the whole model)

The adapter is a small file you can version, review, and load on top of the base model. Merging it into the base
weights (`merge_and_unload()`) gives a standalone model with no adapter overhead at inference, at the cost of a full
copy of the weights.

In [ ]:
import os

trainer.save_model("qwen25-triage-lora/adapter")
size = sum(f.stat().st_size for f in pathlib.Path("qwen25-triage-lora/adapter").rglob("*") if f.is_file())
print(f"adapter directory: {size / 1e6:.1f} MB")

# Optional: a standalone merged model.
# merged = trainer.model.merge_and_unload()
# merged.save_pretrained("qwen25-triage-merged"); tokenizer.save_pretrained("qwen25-triage-merged")

## 9. Optional: the QLoRA variant

To train against a 4-bit NF4 copy of the base model (the QLoRA recipe), pass a `BitsAndBytesConfig` to the trainer.
bitsandbytes runs on NVIDIA CUDA GPUs (and some Intel and CPU backends), not on Apple Silicon; on a Mac, use the MLX
script in this folder instead. For a 0.5B model the memory saving is small, so do this to learn the API and to compare
quality, not because you need it.

In [ ]:
# from transformers import BitsAndBytesConfig
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
# qlora_trainer = SFTTrainer(model=MODEL_ID, args=args, train_dataset=ds["train"], eval_dataset=ds["valid"],
#                            processing_class=tokenizer, peft_config=peft_config, quantization_config=bnb)
# qlora_trainer.train()
# print(evaluate(qlora_trainer.model)[0])

## What to notice

- **The baseline matters.** Compare valid-JSON rate and exact match for base and tuned. A small instruct model can
  often produce some JSON from the system prompt alone; fine-tuning mostly buys consistency of format and labels.
- **Loss is not the metric.** A low eval loss and a high exact-match rate are different claims. You need the second.
- **Held-out templates.** The test set uses sentence templates the model never saw. If you re-split by row instead,
  scores rise, and that rise is leakage, not skill.
- **Forgetting.** Ask the tuned model a general question with no system prompt. Narrow fine-tuning can change
  behavior outside the task; week 4 measures that properly.
- **Record the run.** Model id and revision, dataset hash, LoRA and training settings, seed, and both score tables.
  That is the minimum another engineer needs to reproduce or challenge your result.